# O1-07 — Compatibility Detector Integration

## Objective

Integrate the validated O1 compatibility detector in front of the existing
RYA Gamma neural network.

The integrated workflow will:

1. receive an RYA case;
2. validate the incoming data;
3. calculate the pre-inference screening features;
4. estimate Gamma incompatibility probability;
5. apply the provisional Oct 9 detector threshold of 0.10;
6. route flagged cases to a fallback placeholder;
7. route non-flagged cases to the existing RYA Gamma neural network.

The existing RYA neural network is not retrained or modified.

In [57]:
!pip install onnxruntime

   ---------------------------------------- 0.0/14.9 MB ? eta -:--:--
   --- ------------------------------------ 1.3/14.9 MB 8.4 MB/s eta 0:00:02
   --------- ------------------------------ 3.4/14.9 MB 9.2 MB/s eta 0:00:02
   -------------- ------------------------- 5.2/14.9 MB 9.1 MB/s eta 0:00:02
   ------------------- -------------------- 7.1/14.9 MB 9.1 MB/s eta 0:00:01
   ----------------------- ---------------- 8.7/14.9 MB 8.7 MB/s eta 0:00:01
   ------------------------- -------------- 9.4/14.9 MB 8.0 MB/s eta 0:00:01
   ---------------------------- ----------- 10.7/14.9 MB 7.5 MB/s eta 0:00:01
   ------------------------------- -------- 11.8/14.9 MB 7.1 MB/s eta 0:00:01
   --------------------------------- ------ 12.6/14.9 MB 6.9 MB/s eta 0:00:01
   -------------------------------------- - 14.2/14.9 MB 6.8 MB/s eta 0:00:01
   ---------------------------------------- 14.9/14.9 MB 6.7 MB/s  0:00:02

   -------------------- ------------------- 1/2 [onnxruntime]
   ---------------


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Load the supplied RYA neural-network metadata

The existing RYA neural network is treated as a fixed downstream component.

The supplied metadata is inspected to confirm the exact feature order,
normalisation parameters and output definition before integration.

In [4]:
from pathlib import Path

BASE_DIR = Path(
    r"C:\AIML Y2 S2\Capstone 1\RYA"
)

MODEL_DIR = (
    BASE_DIR
    / "final_network_model"
    / "final_network_model"
)

CODE_DIR = (
    BASE_DIR
    / "final_code"
)

ONNX_FILE = (
    MODEL_DIR
    / "EXP002_nn_optimal_epoch.onnx"
)

METADATA_FILE = (
    MODEL_DIR
    / "EXP002.json"
)

print("ONNX model exists:", ONNX_FILE.exists())
print("Metadata exists:", METADATA_FILE.exists())
print("Code folder exists:", CODE_DIR.exists())

ONNX model exists: True
Metadata exists: True
Code folder exists: True


In [6]:
import json

with open(
    METADATA_FILE,
    "r"
) as f:
    rya_metadata = json.load(f)

print("Number of NN layers:", rya_metadata["n_layers"])
print("Layer width:", rya_metadata["width"])

print(
    "\nNumber of NN input features:",
    len(rya_metadata["feature_labels"])
)

print("\nNN feature order:")
for i, feature in enumerate(
    rya_metadata["feature_labels"],
    start=1
):
    print(f"{i:02d}. {feature}")

print(
    "\nOutput labels:",
    rya_metadata["output_labels"]
)

print(
    "\nP transformation:",
    rya_metadata["P_transform"]
)

Number of NN layers: 20
Layer width: 256

Number of NN input features: 40

NN feature order:
01. X_1
02. X_2
03. X_3
04. X_4
05. Y_1
06. Y_2
07. Y_3
08. Y_4
09. XmY_1
10. XmY_2
11. XmY_3
12. XmY_4
13. mean_X
14. stdev_X
15. skewness_X
16. kurtosis_X
17. mean_Y
18. stdev_Y
19. skewness_Y
20. kurtosis_Y
21. euclid_dist_XY
22. n_X
23. n_Y
24. P_0
25. P_1
26. P_2
27. P_3
28. P_4
29. P_5
30. P_6
31. P_7
32. P_8
33. P_9
34. P_10
35. P_11
36. P_12
37. P_13
38. P_14
39. P_15
40. asymm

Output labels: ['shape', 'rate']

P transformation: all_P_asymm


## 2. Verify the RYA neural-network normalisation metadata

The supplied RYA metadata contains the min/max values used to scale the
40 neural-network input features and to reverse-scale the predicted Gamma
shape and rate outputs.

These values are loaded directly from the supplied `EXP002.json` file so the
existing neural-network behaviour is preserved during integration.

In [9]:
import pandas as pd

nn_minmax = pd.read_json(
    rya_metadata["min_max_df"]
)

print("Min/max table shape:", nn_minmax.shape)

display(
    nn_minmax.head(10)
)

print("\nLast rows:")
display(
    nn_minmax.tail()
)

Min/max table shape: (42, 3)


C:\Users\Willy\AppData\Local\Temp\ipykernel_67652\2818190344.py:3: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  nn_minmax = pd.read_json(


,column,min,max
0,X_1,0.0,0.956522
1,X_2,0.0,0.977778
2,X_3,0.0,0.969697
3,X_4,0.0,0.944444
4,Y_1,0.0,0.976190
5,Y_2,0.0,1.000000
6,Y_3,0.0,1.000000
7,Y_4,0.0,1.000000
8,XmY_1,0.0,0.883653
9,XmY_2,0.0,0.998801



Last rows:


,column,min,max
37,P_14,0.025000,0.306000
38,P_15,0.498000,0.968000
39,asymm,0.000306,0.049194
40,shape,0.669738,2630.171626
41,rate,0.202147,2336.960463


In [11]:
CODE_DIR = (
    BASE_DIR
    / "final_code"
    / "final_code"
)

In [13]:
FEATURE_SCRIPT = (
    CODE_DIR
    / "feature_calculation_portal_script.py"
)

B_MATRIX_FILE = (
    CODE_DIR
    / "B_matrix.txt"
)

print("Feature script exists:", FEATURE_SCRIPT.exists())
print("B matrix exists:", B_MATRIX_FILE.exists())

Feature script exists: True
B matrix exists: True


In [15]:
with open(
    FEATURE_SCRIPT,
    "r"
) as f:
    feature_script_text = f.read()

print(feature_script_text)

import numpy as np
import json 
import pandas as pd

#%%
# Read in P_matrices and network metadata
with open("/Users/liam/Documents/Resilient Youth/Gemfinder/gemfinder/RY_P_matrices/P_matrices_test.json") as f:
    P_matrices = json.load(f)
    
with open("/Users/liam/Documents/Resilient Youth/Gemfinder/gemfinder/d_hat_neural_networks_July22/test_save_networks/network_metadata/EXP003.json") as f:
    metadata = json.load(f)

#%% Necessary functions for calculation

def moment_calculator(prob_dist, dist_name, num_types=4):
    # Takes normalised prob distribution prob_dist on 1:4, i.e. X or Y
    rv_vec = np.arange(1, num_types+1) # i.e. 1,2,3,4
    
    mean = np.dot(rv_vec, prob_dist)
    stdev = np.sqrt(np.dot((rv_vec - mean)**2, prob_dist))
    skewness = np.dot((rv_vec - mean)**3, prob_dist)/stdev**3
    kurtosis = np.dot((rv_vec - mean)**4, prob_dist)/stdev**4
  
    moments = pd.Series({'mean_{}'.format(dist_name): mean,
                         'stdev_{}'.format(dist_name): stde

### RYA portal-script review

The supplied RYA portal script confirms the exact neural-network feature
calculation procedure.

The script was originally written with developer-specific file paths and an
`EXP003.json` metadata reference. For this integration, the feature-generation
logic is preserved, but the supplied `EXP002.json` metadata and
`EXP002_nn_optimal_epoch.onnx` model are used.

The portal script also requires the RYA P-matrix JSON so that a survey question
code can be converted into its corresponding 4x4 unreliability matrix.

In [18]:
from io import StringIO
import numpy as np
import pandas as pd

# Read EXP002 min/max metadata cleanly
nn_minmax = pd.read_json(
    StringIO(rya_metadata["min_max_df"])
)


def rya_moment_calculator(prob_dist, dist_name, num_types=4):
    rv_vec = np.arange(1, num_types + 1)

    mean = np.dot(rv_vec, prob_dist)

    stdev = np.sqrt(
        np.dot(
            (rv_vec - mean) ** 2,
            prob_dist
        )
    )

    # The original RYA code divides directly by stdev.
    # We explicitly return NaN for degenerate distributions
    # so the integration layer can catch them cleanly.
    if stdev == 0:
        skewness = np.nan
        kurtosis = np.nan
    else:
        skewness = (
            np.dot(
                (rv_vec - mean) ** 3,
                prob_dist
            )
            / stdev ** 3
        )

        kurtosis = (
            np.dot(
                (rv_vec - mean) ** 4,
                prob_dist
            )
            / stdev ** 4
        )

    return pd.Series({
        f"mean_{dist_name}": mean,
        f"stdev_{dist_name}": stdev,
        f"skewness_{dist_name}": skewness,
        f"kurtosis_{dist_name}": kurtosis
    })


def rya_asymm_calc(P):
    non_diag_pairs = [
        (0, 1),
        (0, 2),
        (0, 3),
        (1, 2),
        (1, 3),
        (2, 3)
    ]

    asymm_total = 0.0

    for i, j in non_diag_pairs:
        asymm_total += (
            P[i, j] - P[j, i]
        ) ** 2

    return asymm_total / len(non_diag_pairs)


def rya_transform_P_matrix(P_matrix):
    P_matrix = np.asarray(
        P_matrix,
        dtype=float
    )

    if P_matrix.shape != (4, 4):
        raise ValueError(
            f"Expected 4x4 P matrix, got {P_matrix.shape}"
        )

    P_dict = {
        f"P_{i}": P_matrix.flat[i]
        for i in range(16)
    }

    P_dict["asymm"] = rya_asymm_calc(
        P_matrix
    )

    return pd.Series(P_dict)


def build_rya_nn_features(x, y, P_matrix):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if x.shape != (4,) or y.shape != (4,):
        raise ValueError(
            "x and y must each contain exactly four response counts."
        )

    n_X = x.sum()
    n_Y = y.sum()

    if n_X <= 0 or n_Y <= 0:
        raise ValueError(
            "Both response groups must contain at least one observation."
        )

    X = x / n_X
    Y = y / n_Y

    XmY = np.abs(X - Y)

    # Preserve RYA rounding exactly
    euclid_dist_XY = np.round(
        np.sqrt(
            np.sum(
                (X - Y) ** 2
            )
        ),
        4
    )

    X_series = pd.Series({
        f"X_{i + 1}": X[i]
        for i in range(4)
    })

    Y_series = pd.Series({
        f"Y_{i + 1}": Y[i]
        for i in range(4)
    })

    XmY_series = pd.Series({
        f"XmY_{i + 1}": XmY[i]
        for i in range(4)
    })

    X_moments = rya_moment_calculator(
        X,
        "X"
    )

    Y_moments = rya_moment_calculator(
        Y,
        "Y"
    )

    other_features = pd.Series({
        "euclid_dist_XY": euclid_dist_XY,
        "n_X": n_X,
        "n_Y": n_Y
    })

    P_features = rya_transform_P_matrix(
        P_matrix
    )

    features = pd.concat([
        X_series,
        Y_series,
        XmY_series,
        X_moments,
        Y_moments,
        other_features,
        P_features
    ])

    # Force exact EXP002 input order
    features = features.reindex(
        rya_metadata["feature_labels"]
    )

    return features

In [20]:
print("Searching for possible P-matrix JSON files...")

for p in BASE_DIR.rglob("*.json"):
    name = p.name.lower()

    if (
        "matrix" in name
        or "matrices" in name
        or "unreliability" in name
    ):
        print(p)

Searching for possible P-matrix JSON files...
C:\AIML Y2 S2\Capstone 1\RYA\unreliability-matrices.json


## 4. Load the RYA unreliability matrices

The supplied unreliability-matrix JSON is used to map each RYA survey question
code to its corresponding 4x4 P matrix.

Only 4x4 matrices are valid for the current Gamma neural network, which was
trained for four-category response distributions.

In [23]:
P_MATRIX_FILE = (
    BASE_DIR
    / "unreliability-matrices.json"
)

print(
    "P-matrix file exists:",
    P_MATRIX_FILE.exists()
)

P-matrix file exists: True


In [25]:
with open(
    P_MATRIX_FILE,
    "r"
) as f:
    P_matrices = json.load(f)

print(
    "Number of matrices loaded:",
    len(P_matrices)
)

print(
    "Example keys:",
    list(P_matrices.keys())[:10]
)

print(
    "\nry9 exists:",
    "ry9" in P_matrices
)

Number of matrices loaded: 59
Example keys: ['ry2', 'ry3', 'ry4', 'ry5', 'sun5', 'ry6', 'ry7', 'ry9', 'ry8', 'ry10']

ry9 exists: True


In [27]:
ry9_matrix = np.asarray(
    P_matrices["ry9"],
    dtype=float
)

print(
    "ry9 matrix shape:",
    ry9_matrix.shape
)

print("\nry9 matrix:")
print(ry9_matrix)

print(
    "\nRow sums:",
    ry9_matrix.sum(axis=1)
)

ry9 matrix shape: (4, 4)

ry9 matrix:
[[0.43 0.41 0.1  0.06]
 [0.05 0.57 0.28 0.1 ]
 [0.01 0.12 0.61 0.27]
 [0.01 0.02 0.16 0.81]]

Row sums: [1.   1.   1.01 1.  ]


In [29]:
x_example = np.array(
    [40, 52, 7, 14]
)

y_example = np.array(
    [9, 17, 54, 12]
)

question_example = "ry9"

P_example = np.asarray(
    P_matrices[
        question_example
    ],
    dtype=float
)

rya_features_example = (
    build_rya_nn_features(
        x_example,
        y_example,
        P_example
    )
)

print(
    "Number of generated features:",
    len(rya_features_example)
)

print(
    "\nMissing feature values:",
    rya_features_example.isna().sum()
)

print("\nFeature vector:")
display(
    rya_features_example
    .to_frame("value")
)

Number of generated features: 40

Missing feature values: 0

Feature vector:


,value
X_1,0.353982
X_2,0.460177
X_3,0.061947
X_4,0.123894
Y_1,0.097826
Y_2,0.184783
Y_3,0.586957
Y_4,0.130435
XmY_1,0.256156
XmY_2,0.275394


In [31]:
feature_order_matches = (
    rya_features_example.index.tolist()
    == rya_metadata[
        "feature_labels"
    ]
)

print(
    "Feature order matches EXP002 metadata:",
    feature_order_matches
)

Feature order matches EXP002 metadata: True


## 5. Normalise the RYA neural-network inputs

The existing RYA neural network was trained using min-max normalised input
features.

The supplied `EXP002.json` metadata contains the training minimum and maximum
for each of the 40 input variables.

The raw 40-feature vector is therefore normalised using exactly the supplied
metadata before ONNX inference.

In [34]:
def normalise_rya_nn_features(
    raw_features,
    minmax_df
):
    normalised = raw_features.copy().astype(float)

    for feature_name in raw_features.index:

        row = minmax_df[
            minmax_df["column"]
            == feature_name
        ]

        if len(row) != 1:
            raise ValueError(
                f"Expected exactly one min/max row "
                f"for {feature_name}, found {len(row)}."
            )

        fmin = float(
            row["min"].iloc[0]
        )

        fmax = float(
            row["max"].iloc[0]
        )

        if fmax == fmin:
            raise ValueError(
                f"Invalid min/max range for "
                f"{feature_name}: {fmin} to {fmax}"
            )

        normalised.loc[
            feature_name
        ] = (
            raw_features.loc[feature_name]
            - fmin
        ) / (
            fmax - fmin
        )

    return normalised

In [36]:
rya_features_norm = (
    normalise_rya_nn_features(
        rya_features_example,
        nn_minmax
    )
)

print(
    "Normalised feature count:",
    len(rya_features_norm)
)

print(
    "Missing normalised values:",
    rya_features_norm.isna().sum()
)

print(
    "Infinite normalised values:",
    np.isinf(
        rya_features_norm.to_numpy()
    ).sum()
)

display(
    rya_features_norm
    .to_frame("normalised_value")
)

Normalised feature count: 40
Missing normalised values: 0
Infinite normalised values: 0


,normalised_value
X_1,0.370072
X_2,0.470636
X_3,0.063883
X_4,0.131182
Y_1,0.100212
Y_2,0.184783
Y_3,0.586957
Y_4,0.130435
XmY_1,0.289883
XmY_2,0.275725


In [38]:
out_of_range = rya_features_norm[
    (rya_features_norm < 0)
    | (rya_features_norm > 1)
]

print(
    "Features outside training min/max range:",
    len(out_of_range)
)

display(
    out_of_range.to_frame(
        "normalised_value"
    )
)

Features outside training min/max range: 0


,normalised_value


## 6. Inspect the supplied ONNX model interface

The supplied RYA ONNX model is loaded directly and its input/output interface is
inspected before inference.

This avoids hard-coding ONNX tensor names or shapes.

In [41]:
BASE_DIR
MODEL_DIR
ONNX_FILE
METADATA_FILE
rya_metadata
nn_minmax

,column,min,max
0,X_1,0.000000,0.956522
1,X_2,0.000000,0.977778
2,X_3,0.000000,0.969697
3,X_4,0.000000,0.944444
4,Y_1,0.000000,0.976190
5,Y_2,0.000000,1.000000
6,Y_3,0.000000,1.000000
7,Y_4,0.000000,1.000000
8,XmY_1,0.000000,0.883653
9,XmY_2,0.000000,0.998801


In [43]:
import onnxruntime as ort

session = ort.InferenceSession(
    str(ONNX_FILE)
)

print("ONNX inputs:")
for inp in session.get_inputs():
    print(
        "name =", inp.name,
        "| shape =", inp.shape,
        "| type =", inp.type
    )

print("\nONNX outputs:")
for out in session.get_outputs():
    print(
        "name =", out.name,
        "| shape =", out.shape,
        "| type =", out.type
    )

ONNX inputs:
name = input.1 | shape = [40] | type = tensor(float)

ONNX outputs:
name = 125 | shape = [2] | type = tensor(float)


## 7. Run the supplied RYA ONNX neural network

The validated and normalised 40-feature vector is converted to the exact
float32 tensor format expected by the supplied ONNX model.

The model returns two normalised outputs corresponding to Gamma `shape` and
`rate`.

These outputs are then reverse-normalised using the min/max values supplied in
`EXP002.json`.

In [46]:
# Convert the 40 normalised features to the exact ONNX input format
onnx_input = (
    rya_features_norm
    .to_numpy(dtype=np.float32)
)

print("Input shape:", onnx_input.shape)
print("Input dtype:", onnx_input.dtype)
print("Contains NaN:", np.isnan(onnx_input).any())
print("Contains inf:", np.isinf(onnx_input).any())

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

onnx_output = session.run(
    [output_name],
    {
        input_name: onnx_input
    }
)[0]

print("\nRaw ONNX output:")
print(onnx_output)

print(
    "\nOutput shape:",
    onnx_output.shape
)

print(
    "Output dtype:",
    onnx_output.dtype
)

Input shape: (40,)
Input dtype: float32
Contains NaN: False
Contains inf: False

Raw ONNX output:
[0.00353707 0.00886882]

Output shape: (2,)
Output dtype: float32


In [48]:
def reverse_normalise_value(
    normalised_value,
    feature_name,
    minmax_df
):
    row = minmax_df[
        minmax_df["column"] == feature_name
    ]

    if len(row) != 1:
        raise ValueError(
            f"Expected one min/max row for "
            f"{feature_name}, found {len(row)}."
        )

    fmin = float(
        row["min"].iloc[0]
    )

    fmax = float(
        row["max"].iloc[0]
    )

    return (
        (fmax - fmin)
        * float(normalised_value)
        + fmin
    )


shape_pred = reverse_normalise_value(
    onnx_output[0],
    "shape",
    nn_minmax
)

rate_pred = reverse_normalise_value(
    onnx_output[1],
    "rate",
    nn_minmax
)

print("Predicted Gamma shape:", shape_pred)
print("Predicted Gamma rate:", rate_pred)

Predicted Gamma shape: 9.970475608021188
Predicted Gamma rate: 20.926446972942113


In [50]:
rya_nn_result = {
    "question": question_example,
    "n_X": int(x_example.sum()),
    "n_Y": int(y_example.sum()),
    "gamma_shape": shape_pred,
    "gamma_rate": rate_pred
}

rya_nn_result

{'question': 'ry9',
 'n_X': 113,
 'n_Y': 92,
 'gamma_shape': 9.970475608021188,
 'gamma_rate': 20.926446972942113}

### RYA neural-network reproduction checkpoint

The supplied RYA neural-network pathway was reproduced successfully.

For the example `ry9` case, the integration notebook:

1. generated the exact 40 features expected by `EXP002.json`;
2. matched the required feature order;
3. applied the supplied min-max normalisation;
4. passed a float32 40-element vector into the supplied ONNX model;
5. reverse-normalised the two network outputs.

The resulting Gamma parameters were:

- shape = 9.9705
- rate = 20.9264

This confirms that the existing RYA Gamma neural network can be called
successfully from the integration notebook without retraining or modifying
the model.

## 8. Prepare the O1 compatibility detector for integration

The existing RYA Gamma neural network uses 40 features, while the O1
compatibility detector uses a separate set of 54 pre-inference screening
features.

Before integrating the detector, the original O1 screening-feature extraction
implementation is located and reused so that the production integration uses
the same feature definitions as the Sep 25 and Oct 9 experiments.

The detector and neural-network feature paths remain separate because they
serve different purposes.

In [54]:
print("Searching for O1 screening feature extractor...")

feature_extractor_matches = list(
    BASE_DIR.rglob("feature_extractor.py")
)

for p in feature_extractor_matches:
    print(p)

print(
    "\nNumber found:",
    len(feature_extractor_matches)
)

Searching for O1 screening feature extractor...
C:\AIML Y2 S2\Capstone 1\RYA\feature_extractor.py

Number found: 1


In [56]:
print("Possible related Python files:")

for p in BASE_DIR.rglob("*.py"):
    name = p.name.lower()

    if (
        "feature" in name
        or "screen" in name
        or "detector" in name
    ):
        print(p)

Possible related Python files:
C:\AIML Y2 S2\Capstone 1\RYA\feature_extractor.py
C:\AIML Y2 S2\Capstone 1\RYA\final_code\final_code\feature_calculation_portal_script.py


In [58]:
FEATURE_EXTRACTOR_FILE = (
    BASE_DIR
    / "feature_extractor.py"
)

with open(
    FEATURE_EXTRACTOR_FILE,
    "r"
) as f:
    screening_feature_code = f.read()

print(screening_feature_code)

from __future__ import annotations

import re
import numpy as np
import pandas as pd


def parse_r_c_vector(text: str) -> np.ndarray:
    """
    Parse strings such as:
    c(0.63, 0.149, ..., 0.77)

    into a NumPy matrix.

    The RYA instance files store P_matrix values in R-style c(...)
    format. R uses column-major ordering, so reshape with order="F".
    """
    if isinstance(text, np.ndarray):
        arr = text.astype(float)
    elif isinstance(text, (list, tuple)):
        arr = np.asarray(text, dtype=float)
    else:
        nums = [
            float(x)
            for x in re.findall(
                r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?",
                str(text)
            )
        ]
        arr = np.asarray(nums, dtype=float)

    if arr.size == 16:
        return arr.reshape((4, 4), order="F")
    if arr.size == 36:
        return arr.reshape((6, 6), order="F")

    raise ValueError(f"Unsupported P_matrix size: {arr.size}")


def entropy(p: np.ndarray, eps: float = 1e

### 8.2 Verify the screening feature extractor against the training data

Before integrating the compatibility detector, the production-style screening
feature function is checked against the previously saved classifier input data.

For a known RYA instance, the 54 features are recalculated from X, Y, group
sizes and the question-specific P matrix.

The recalculated values are then compared with the features used during the
Sep 25 and Oct 9 detector experiments.

This verifies that detector integration uses exactly the same feature
definitions as detector training.

In [61]:
import sys

if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

print(str(BASE_DIR) in sys.path)

True


In [63]:
from feature_extractor import extract_screening_features

print("Feature extractor imported successfully.")

Feature extractor imported successfully.


In [65]:
from feature_extractor import extract_screening_features

SCREENING_DATA_FILE = (
    BASE_DIR
    / "O1_classifier_inputs_unlabelled.csv"
)

screening_data = pd.read_csv(
    SCREENING_DATA_FILE
)

print(
    "Screening dataset:",
    screening_data.shape
)

print(
    "Feature extractor imported successfully."
)

Screening dataset: (63328, 57)
Feature extractor imported successfully.


In [67]:
screening_identifier_cols = [
    "exp_code",
    "P_matrix_question",
    "source_file"
]

screening_feature_cols = [
    col
    for col in screening_data.columns
    if col not in screening_identifier_cols
]

print(
    "Number of detector features:",
    len(screening_feature_cols)
)

print("\nDetector feature order:")

for i, col in enumerate(
    screening_feature_cols,
    start=1
):
    print(f"{i:02d}. {col}")

Number of detector features: 54

Detector feature order:
01. n_X
02. n_Y
03. log_n_X
04. log_n_Y
05. group_size_ratio
06. log_group_size_ratio
07. X_1
08. X_2
09. X_3
10. X_4
11. Y_1
12. Y_2
13. Y_3
14. Y_4
15. euclid_dist_XY
16. absdiff_1
17. absdiff_2
18. absdiff_3
19. absdiff_4
20. min_prop_X
21. min_prop_Y
22. max_prop_X
23. max_prop_Y
24. active_categories_X
25. active_categories_Y
26. degenerate_X
27. degenerate_Y
28. sparse_count_X
29. sparse_count_Y
30. entropy_X
31. entropy_Y
32. mean_diag_P
33. min_diag_P
34. max_diag_P
35. mean_offdiag_mass_P
36. asymmetry_P
37. max_row_sum_error_P
38. all_rows_diagonal_mode_P
39. P_11
40. P_12
41. P_13
42. P_14
43. P_21
44. P_22
45. P_23
46. P_24
47. P_31
48. P_32
49. P_33
50. P_34
51. P_41
52. P_42
53. P_43
54. P_44


In [69]:
test_row = screening_data.iloc[0]

test_question = (
    test_row["P_matrix_question"]
)

test_P = np.asarray(
    P_matrices[test_question],
    dtype=float
)

test_X = np.array(
    [
        test_row[f"X_{i}"]
        for i in range(1, 5)
    ],
    dtype=float
)

test_Y = np.array(
    [
        test_row[f"Y_{i}"]
        for i in range(1, 5)
    ],
    dtype=float
)

recomputed_screening = (
    extract_screening_features(
        X=test_X,
        Y=test_Y,
        n_x=test_row["n_X"],
        n_y=test_row["n_Y"],
        P=test_P
    )
)

recomputed_screening = pd.Series(
    recomputed_screening
).reindex(
    screening_feature_cols
)

stored_screening = (
    test_row[
        screening_feature_cols
    ]
    .astype(float)
)

In [71]:
comparison = pd.DataFrame({
    "stored": stored_screening,
    "recomputed": recomputed_screening
})

comparison["abs_difference"] = (
    comparison["stored"]
    - comparison["recomputed"]
).abs()

print(
    "Test exp_code:",
    test_row["exp_code"]
)

print(
    "Question:",
    test_question
)

print(
    "Maximum absolute difference:",
    comparison[
        "abs_difference"
    ].max()
)

print(
    "Features differing by > 1e-10:",
    (
        comparison[
            "abs_difference"
        ] > 1e-10
    ).sum()
)

display(
    comparison.sort_values(
        "abs_difference",
        ascending=False
    ).head(10)
)

Test exp_code: orthog_B_3_large_classes_1
Question: cyb1
Maximum absolute difference: 0.016
Features differing by > 1e-10: 22


,stored,recomputed,abs_difference
P_24,6.400000e-02,0.08,0.016
min_diag_P,6.350000e-01,0.62,0.015
P_22,6.350000e-01,0.62,0.015
P_13,7.000000e-02,0.06,0.010
max_row_sum_error_P,1.110223e-16,0.01,0.010
P_11,8.220000e-01,0.83,0.008
P_33,7.360000e-01,0.73,0.006
P_21,8.600000e-02,0.08,0.006
max_diag_P,8.750000e-01,0.88,0.005
P_23,2.150000e-01,0.21,0.005


In [73]:
test_row = screening_data.iloc[0]

test_X = np.array(
    [
        test_row[f"X_{i}"]
        for i in range(1, 5)
    ],
    dtype=float
)

test_Y = np.array(
    [
        test_row[f"Y_{i}"]
        for i in range(1, 5)
    ],
    dtype=float
)

# Reconstruct the exact P matrix used to create
# the saved detector-training features
test_P_stored = np.array([
    [
        test_row["P_11"],
        test_row["P_12"],
        test_row["P_13"],
        test_row["P_14"]
    ],
    [
        test_row["P_21"],
        test_row["P_22"],
        test_row["P_23"],
        test_row["P_24"]
    ],
    [
        test_row["P_31"],
        test_row["P_32"],
        test_row["P_33"],
        test_row["P_34"]
    ],
    [
        test_row["P_41"],
        test_row["P_42"],
        test_row["P_43"],
        test_row["P_44"]
    ]
], dtype=float)

print("Stored P matrix:")
print(test_P_stored)

print("\nRow sums:")
print(test_P_stored.sum(axis=1))

Stored P matrix:
[[0.822 0.071 0.07  0.037]
 [0.086 0.635 0.215 0.064]
 [0.03  0.066 0.736 0.168]
 [0.023 0.021 0.081 0.875]]

Row sums:
[1. 1. 1. 1.]


In [75]:
recomputed_screening = extract_screening_features(
    X=test_X,
    Y=test_Y,
    n_x=test_row["n_X"],
    n_y=test_row["n_Y"],
    P=test_P_stored
)

recomputed_screening = pd.Series(
    recomputed_screening
).reindex(
    screening_feature_cols
)

stored_screening = (
    test_row[
        screening_feature_cols
    ]
    .astype(float)
)

comparison_exact_P = pd.DataFrame({
    "stored": stored_screening,
    "recomputed": recomputed_screening
})

comparison_exact_P["abs_difference"] = (
    comparison_exact_P["stored"]
    - comparison_exact_P["recomputed"]
).abs()

print(
    "Maximum absolute difference:",
    comparison_exact_P[
        "abs_difference"
    ].max()
)

print(
    "Features differing by > 1e-10:",
    (
        comparison_exact_P[
            "abs_difference"
        ] > 1e-10
    ).sum()
)

display(
    comparison_exact_P.sort_values(
        "abs_difference",
        ascending=False
    ).head(10)
)

Maximum absolute difference: 4.3191725812408155e-05
Features differing by > 1e-10: 1


,stored,recomputed,abs_difference
euclid_dist_XY,2.807000e-01,2.806568e-01,4.319173e-05
entropy_X,1.359012e+00,1.359012e+00,2.220446e-16
group_size_ratio,1.580956e+00,1.580956e+00,2.220446e-16
log_group_size_ratio,4.580294e-01,4.580294e-01,5.551115e-17
absdiff_2,1.757785e-01,1.757785e-01,5.551115e-17
absdiff_1,9.743450e-02,9.743450e-02,1.387779e-17
max_row_sum_error_P,1.110223e-16,1.110223e-16,2.465190e-32
P_41,2.300000e-02,2.300000e-02,0.000000e+00
P_34,1.680000e-01,1.680000e-01,0.000000e+00
mean_diag_P,7.670000e-01,7.670000e-01,0.000000e+00


### Screening-feature reproduction finding

The O1 screening feature extractor was verified against a previously saved
detector-training instance.

Using the exact P matrix stored with the training instance, 53 of the 54
features reproduced to numerical precision.

The only difference was `euclid_dist_XY`, where the stored value was rounded
to four decimal places:

- stored = 0.2807;
- direct recomputation = 0.2806568.

This is consistent with the original RYA portal implementation, which rounds
Euclidean distance to four decimal places.

The integration implementation will therefore preserve this rounding so that
live detector inputs remain consistent with the features used during detector
development.

In [78]:
def build_screening_features(
    X,
    Y,
    n_x,
    n_y,
    P_matrix,
    feature_order
):
    """
    Build the 54 O1 detector features using the same
    definitions used during Sep 25 / Oct 9 development.
    """

    features = extract_screening_features(
        X=X,
        Y=Y,
        n_x=n_x,
        n_y=n_y,
        P=P_matrix
    )

    features = pd.Series(
        features,
        dtype=float
    )

    # Preserve training / RYA portal convention
    features["euclid_dist_XY"] = np.round(
        features["euclid_dist_XY"],
        4
    )

    features = features.reindex(
        feature_order
    )

    if features.isna().any():
        missing = features[
            features.isna()
        ].index.tolist()

        raise ValueError(
            f"Missing screening features: {missing}"
        )

    return features

In [80]:
production_screening = (
    build_screening_features(
        X=test_X,
        Y=test_Y,
        n_x=test_row["n_X"],
        n_y=test_row["n_Y"],
        P_matrix=test_P_stored,
        feature_order=screening_feature_cols
    )
)

comparison_production = pd.DataFrame({
    "stored": stored_screening,
    "production": production_screening
})

comparison_production[
    "abs_difference"
] = (
    comparison_production["stored"]
    - comparison_production["production"]
).abs()

print(
    "Maximum absolute difference:",
    comparison_production[
        "abs_difference"
    ].max()
)

print(
    "Features differing by > 1e-10:",
    (
        comparison_production[
            "abs_difference"
        ] > 1e-10
    ).sum()
)

Maximum absolute difference: 2.220446049250313e-16
Features differing by > 1e-10: 0


In [82]:
verification_rows = (
    screening_data
    .sample(
        n=100,
        random_state=42
    )
)

verification_results = []

for _, row in verification_rows.iterrows():

    X = np.array(
        [row[f"X_{i}"] for i in range(1, 5)],
        dtype=float
    )

    Y = np.array(
        [row[f"Y_{i}"] for i in range(1, 5)],
        dtype=float
    )

    P = np.array([
        [
            row["P_11"],
            row["P_12"],
            row["P_13"],
            row["P_14"]
        ],
        [
            row["P_21"],
            row["P_22"],
            row["P_23"],
            row["P_24"]
        ],
        [
            row["P_31"],
            row["P_32"],
            row["P_33"],
            row["P_34"]
        ],
        [
            row["P_41"],
            row["P_42"],
            row["P_43"],
            row["P_44"]
        ]
    ], dtype=float)

    recomputed = build_screening_features(
        X=X,
        Y=Y,
        n_x=row["n_X"],
        n_y=row["n_Y"],
        P_matrix=P,
        feature_order=screening_feature_cols
    )

    stored = (
        row[screening_feature_cols]
        .astype(float)
    )

    max_diff = (
        stored - recomputed
    ).abs().max()

    verification_results.append({
        "exp_code": row["exp_code"],
        "max_abs_difference": max_diff
    })


verification_df = pd.DataFrame(
    verification_results
)

print(
    "Cases checked:",
    len(verification_df)
)

print(
    "Overall maximum difference:",
    verification_df[
        "max_abs_difference"
    ].max()
)

print(
    "Cases with difference > 1e-10:",
    (
        verification_df[
            "max_abs_difference"
        ] > 1e-10
    ).sum()
)

Cases checked: 100
Overall maximum difference: 3.552713678800501e-15
Cases with difference > 1e-10: 0


## 9. Train and save the integrated O1 compatibility detector

The production screening-feature implementation has been verified against the
detector-training data.

The final Oct 9 Logistic Regression detector is now retrained using all 477
high-confidence labelled cases:

- 333 compatible;
- 144 incompatible;
- 54 pre-inference screening features.

The detector uses the provisional high-recall operating threshold of 0.10.

In [85]:
EXPANDED_DETECTOR_FILE = (
    BASE_DIR
    / "O1_06_detector_training_high_conf_477.csv"
)

detector_expanded = pd.read_csv(
    EXPANDED_DETECTOR_FILE
)

print(
    "Loaded detector dataset:",
    detector_expanded.shape
)

print(
    detector_expanded[
        "compatibility_zone_v1"
    ].value_counts()
)

Loaded detector dataset: (477, 58)
compatibility_zone_v1
compatible      333
incompatible    144
Name: count, dtype: int64


In [87]:
identifier_cols = [
    "exp_code",
    "P_matrix_question",
    "source_file"
]

target_col = "compatibility_zone_v1"

feature_cols = [
    col
    for col in detector_expanded.columns
    if col not in identifier_cols + [target_col]
]

X_expanded = (
    detector_expanded[
        feature_cols
    ]
    .copy()
)

y_expanded = (
    detector_expanded[
        target_col
    ]
    .map({
        "compatible": 0,
        "incompatible": 1
    })
)

print(
    "X_expanded shape:",
    X_expanded.shape
)

print(
    "y_expanded shape:",
    y_expanded.shape
)

print(
    "\nTarget counts:"
)
print(
    y_expanded.value_counts()
)

print(
    "\nFeature order matches screening integration:",
    feature_cols == screening_feature_cols
)

X_expanded shape: (477, 54)
y_expanded shape: (477,)

Target counts:
compatibility_zone_v1
0    333
1    144
Name: count, dtype: int64

Feature order matches screening integration: True


In [89]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

FINAL_DETECTOR_THRESHOLD = 0.10

final_detector = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "classifier",
        LogisticRegression(
            class_weight="balanced",
            max_iter=5000,
            random_state=42
        )
    )
])

final_detector.fit(
    X_expanded,
    y_expanded
)

print("Final detector trained.")
print("Training cases:", len(X_expanded))
print("Features:", X_expanded.shape[1])
print("Threshold:", FINAL_DETECTOR_THRESHOLD)

Final detector trained.
Training cases: 477
Features: 54
Threshold: 0.1


In [91]:
import joblib
import json

DETECTOR_MODEL_FILE = (
    BASE_DIR
    / "O1_07_compatibility_detector_v1.joblib"
)

DETECTOR_METADATA_FILE = (
    BASE_DIR
    / "O1_07_compatibility_detector_v1_metadata.json"
)

joblib.dump(
    final_detector,
    DETECTOR_MODEL_FILE
)

detector_metadata = {
    "model_type": "LogisticRegression",
    "feature_count": int(len(feature_cols)),
    "feature_order": list(feature_cols),
    "threshold": float(FINAL_DETECTOR_THRESHOLD),
    "training_cases": int(len(X_expanded)),
    "compatible_cases": int((y_expanded == 0).sum()),
    "incompatible_cases": int((y_expanded == 1).sum())
}

with open(
    DETECTOR_METADATA_FILE,
    "w"
) as f:
    json.dump(
        detector_metadata,
        f,
        indent=2
    )

print("Saved detector:")
print(DETECTOR_MODEL_FILE)

print("\nSaved metadata:")
print(DETECTOR_METADATA_FILE)

Saved detector:
C:\AIML Y2 S2\Capstone 1\RYA\O1_07_compatibility_detector_v1.joblib

Saved metadata:
C:\AIML Y2 S2\Capstone 1\RYA\O1_07_compatibility_detector_v1_metadata.json


In [93]:
loaded_detector = joblib.load(
    DETECTOR_MODEL_FILE
)

with open(
    DETECTOR_METADATA_FILE,
    "r"
) as f:
    loaded_detector_metadata = json.load(f)

print(
    "Loaded feature count:",
    loaded_detector_metadata["feature_count"]
)

print(
    "Loaded threshold:",
    loaded_detector_metadata["threshold"]
)

print(
    "Feature order matches:",
    loaded_detector_metadata["feature_order"]
    == feature_cols
)

Loaded feature count: 54
Loaded threshold: 0.1
Feature order matches: True


## 10. Integrate the O1 detector with the existing RYA Gamma NN

The validated O1 compatibility detector is now placed before the existing
RYA Gamma neural network.

For each incoming case, the integration performs:

1. raw input validation;
2. 54-feature compatibility screening;
3. incompatibility probability estimation;
4. routing using the provisional threshold of 0.10;
5. Gamma NN inference only for cases not routed to fallback.

The existing RYA neural network is not modified or retrained.

In [96]:
import inspect

print("Screening wrapper:")
print(
    inspect.signature(
        build_screening_features
    )
)

print("\nRYA NN feature builder:")
print(
    inspect.signature(
        build_rya_nn_features
    )
)

print("\nAvailable ONNX-related variables:")

for name in [
    "onnx_session",
    "session",
    "rya_session",
    "ONNX_MODEL_FILE",
    "MODEL_FILE",
    "onnx_model_path"
]:
    if name in globals():
        print(
            name,
            "=",
            globals()[name]
        )

Screening wrapper:
(X, Y, n_x, n_y, P_matrix, feature_order)

RYA NN feature builder:
(x, y, P_matrix)

Available ONNX-related variables:
session = <onnxruntime.capi.onnxruntime_inference_collection.InferenceSession object at 0x00000289812AB440>


In [98]:
print("NN metadata columns:")
print(nn_minmax.columns.tolist())

print("\nNN feature labels:")
print(rya_metadata["feature_labels"])

print(
    "\nNumber of NN features:",
    len(
        rya_metadata[
            "feature_labels"
        ]
    )
)

NN metadata columns:
['column', 'min', 'max']

NN feature labels:
['X_1', 'X_2', 'X_3', 'X_4', 'Y_1', 'Y_2', 'Y_3', 'Y_4', 'XmY_1', 'XmY_2', 'XmY_3', 'XmY_4', 'mean_X', 'stdev_X', 'skewness_X', 'kurtosis_X', 'mean_Y', 'stdev_Y', 'skewness_Y', 'kurtosis_Y', 'euclid_dist_XY', 'n_X', 'n_Y', 'P_0', 'P_1', 'P_2', 'P_3', 'P_4', 'P_5', 'P_6', 'P_7', 'P_8', 'P_9', 'P_10', 'P_11', 'P_12', 'P_13', 'P_14', 'P_15', 'asymm']

Number of NN features: 40


In [100]:
import inspect

print("Screening wrapper:")
print(
    inspect.signature(
        build_screening_features
    )
)

print("\nRYA NN feature builder:")
print(
    inspect.signature(
        build_rya_nn_features
    )
)

print("\nAvailable ONNX-related variables:")

for name in [
    "onnx_session",
    "session",
    "rya_session",
    "ONNX_MODEL_FILE",
    "MODEL_FILE",
    "onnx_model_path"
]:
    if name in globals():
        print(
            name,
            "=",
            globals()[name]
        )

Screening wrapper:
(X, Y, n_x, n_y, P_matrix, feature_order)

RYA NN feature builder:
(x, y, P_matrix)

Available ONNX-related variables:
session = <onnxruntime.capi.onnxruntime_inference_collection.InferenceSession object at 0x00000289812AB440>


### 10.1 Integrated O1 routing function

The following function combines the O1 compatibility detector with the
existing RYA Gamma neural network.

Routing logic:

- invalid inputs are rejected before inference;
- cases with incompatibility probability greater than or equal to 0.10 are
  routed to `fallback_required`;
- cases below the threshold are passed to the unchanged RYA Gamma neural
  network;
- Gamma shape and rate are returned only when the neural-network route is used.

In [103]:
def run_integrated_rya_case(
    x,
    y,
    question,
    P_matrix,
    detector=loaded_detector,
    detector_metadata=loaded_detector_metadata,
    nn_session=session
):
    """
    Integrated O1 compatibility screening + existing RYA Gamma NN routing.

    Pipeline:
        raw input
        -> validation
        -> 54-feature compatibility detector
        -> fallback_required OR existing 40-feature RYA NN
        -> Gamma shape/rate
    """

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    P_matrix = np.asarray(P_matrix, dtype=float)

    validation_warnings = []

    # -------------------------------------------------
    # 1. Basic input validation
    # -------------------------------------------------

    if x.shape != (4,) or y.shape != (4,):
        return {
            "route": "invalid_input",
            "reason": "x and y must each contain 4 category counts."
        }

    if P_matrix.shape != (4, 4):
        return {
            "route": "invalid_input",
            "reason": "P_matrix must be 4x4."
        }

    if (
        np.isnan(x).any()
        or np.isnan(y).any()
        or np.isnan(P_matrix).any()
    ):
        return {
            "route": "invalid_input",
            "reason": "Input contains NaN values."
        }

    if (
        np.isinf(x).any()
        or np.isinf(y).any()
        or np.isinf(P_matrix).any()
    ):
        return {
            "route": "invalid_input",
            "reason": "Input contains infinite values."
        }

    if np.any(x < 0) or np.any(y < 0):
        return {
            "route": "invalid_input",
            "reason": "Negative category counts are not allowed."
        }

    n_x = float(x.sum())
    n_y = float(y.sum())

    if n_x <= 0 or n_y <= 0:
        return {
            "route": "invalid_input",
            "reason": "Both groups must contain at least one observation."
        }

    if np.any(P_matrix < 0):
        return {
            "route": "invalid_input",
            "reason": "P_matrix contains negative probabilities."
        }

    # -------------------------------------------------
    # 2. Validate P-matrix row sums
    # -------------------------------------------------

    row_sums = P_matrix.sum(axis=1)

    max_row_sum_error = float(
        np.max(
            np.abs(
                row_sums - 1.0
            )
        )
    )

    # Some supplied RYA matrices are rounded to
    # two decimal places, so a row may sum to
    # approximately 0.99 or 1.01.
    #
    # Small rounding deviations are accepted,
    # but the original matrix is NOT renormalised
    # because the existing RYA NN uses the supplied
    # values directly.
    P_ROW_SUM_TOLERANCE = 0.020001

    if max_row_sum_error > P_ROW_SUM_TOLERANCE:
        return {
            "route": "invalid_input",
            "reason": (
                "P_matrix row sums deviate too far from 1. "
                f"Maximum error = {max_row_sum_error:.6f}"
            )
        }

    if max_row_sum_error > 1e-6:
        validation_warnings.append(
            (
                "P_matrix row sums contain small rounding deviations "
                f"(maximum error = {max_row_sum_error:.6f}). "
                "Original matrix retained without renormalisation."
            )
        )

    # -------------------------------------------------
    # 3. Convert raw counts to proportions
    # -------------------------------------------------

    X = x / n_x
    Y = y / n_y

    # -------------------------------------------------
    # 4. Build the 54 O1 screening features
    # -------------------------------------------------

    detector_feature_order = (
        detector_metadata["feature_order"]
    )

    screening_features = build_screening_features(
        X=X,
        Y=Y,
        n_x=n_x,
        n_y=n_y,
        P_matrix=P_matrix,
        feature_order=detector_feature_order
    )

    detector_input = pd.DataFrame(
        [
            screening_features.values
        ],
        columns=detector_feature_order
    )

    # -------------------------------------------------
    # 5. Run O1 compatibility detector
    # -------------------------------------------------

    incompatibility_probability = float(
        detector.predict_proba(
            detector_input
        )[0, 1]
    )

    threshold = float(
        detector_metadata["threshold"]
    )

    # -------------------------------------------------
    # 6. Route risky case to fallback
    # -------------------------------------------------

    if incompatibility_probability >= threshold:

        return {
            "question": question,
            "n_X": int(n_x),
            "n_Y": int(n_y),
            "route": "fallback_required",
            "incompatibility_probability":
                incompatibility_probability,
            "threshold": threshold,
            "validation_warnings":
                validation_warnings,
            "gamma_shape": None,
            "gamma_rate": None
        }

    # -------------------------------------------------
    # 7. Build existing 40-feature RYA NN vector
    # -------------------------------------------------

    nn_features = build_rya_nn_features(
        x=x,
        y=y,
        P_matrix=P_matrix
    )

    nn_feature_order = (
        rya_metadata["feature_labels"]
    )

    nn_features = nn_features.reindex(
        nn_feature_order
    )

    # -------------------------------------------------
    # 8. Guard against invalid NN feature values
    # -------------------------------------------------

    if nn_features.isna().any():

        return {
            "question": question,
            "n_X": int(n_x),
            "n_Y": int(n_y),
            "route": "fallback_required",
            "reason": (
                "RYA NN features contain NaN values."
            ),
            "incompatibility_probability":
                incompatibility_probability,
            "threshold": threshold,
            "validation_warnings":
                validation_warnings,
            "gamma_shape": None,
            "gamma_rate": None
        }

    if np.isinf(
        nn_features.values
    ).any():

        return {
            "question": question,
            "n_X": int(n_x),
            "n_Y": int(n_y),
            "route": "fallback_required",
            "reason": (
                "RYA NN features contain infinite values."
            ),
            "incompatibility_probability":
                incompatibility_probability,
            "threshold": threshold,
            "validation_warnings":
                validation_warnings,
            "gamma_shape": None,
            "gamma_rate": None
        }

    # -------------------------------------------------
    # 9. Min-max normalise the 40 NN features
    # -------------------------------------------------

    nn_normalised = []

    for feature_name in nn_feature_order:

        row = nn_minmax[
            nn_minmax["column"]
            == feature_name
        ]

        if len(row) != 1:
            return {
                "question": question,
                "n_X": int(n_x),
                "n_Y": int(n_y),
                "route": "fallback_required",
                "reason": (
                    "Could not find a unique normalisation "
                    f"row for NN feature: {feature_name}"
                ),
                "incompatibility_probability":
                    incompatibility_probability,
                "threshold": threshold,
                "validation_warnings":
                    validation_warnings,
                "gamma_shape": None,
                "gamma_rate": None
            }

        fmin = float(
            row["min"].iloc[0]
        )

        fmax = float(
            row["max"].iloc[0]
        )

        value = float(
            nn_features[feature_name]
        )

        if fmax == fmin:
            norm_value = 0.0

        else:
            norm_value = (
                value - fmin
            ) / (
                fmax - fmin
            )

        nn_normalised.append(
            norm_value
        )

    nn_input = np.asarray(
        nn_normalised,
        dtype=np.float32
    )

    # -------------------------------------------------
    # 10. Final NN-input safety check
    # -------------------------------------------------

    if (
        np.isnan(nn_input).any()
        or np.isinf(nn_input).any()
    ):
        return {
            "question": question,
            "n_X": int(n_x),
            "n_Y": int(n_y),
            "route": "fallback_required",
            "reason": (
                "Normalised RYA NN input contains "
                "NaN or infinite values."
            ),
            "incompatibility_probability":
                incompatibility_probability,
            "threshold": threshold,
            "validation_warnings":
                validation_warnings,
            "gamma_shape": None,
            "gamma_rate": None
        }

    # -------------------------------------------------
    # 11. Run existing RYA ONNX neural network
    # -------------------------------------------------

    input_name = (
        nn_session
        .get_inputs()[0]
        .name
    )

    raw_output = nn_session.run(
        None,
        {
            input_name: nn_input
        }
    )[0]

    raw_output = np.asarray(
        raw_output
    ).reshape(-1)

    # -------------------------------------------------
    # 12. Reverse-normalise Gamma outputs
    # -------------------------------------------------

    shape_pred = reverse_normalise_value(
        raw_output[0],
        "shape",
        nn_minmax
    )

    rate_pred = reverse_normalise_value(
        raw_output[1],
        "rate",
        nn_minmax
    )

    # -------------------------------------------------
    # 13. Final integrated output
    # -------------------------------------------------

    return {
        "question": question,
        "n_X": int(n_x),
        "n_Y": int(n_y),
        "route": "use_gamma_nn",
        "incompatibility_probability":
            incompatibility_probability,
        "threshold": threshold,
        "validation_warnings":
            validation_warnings,
        "gamma_shape": float(shape_pred),
        "gamma_rate": float(rate_pred)
    }

In [105]:
integrated_result_ry9 = run_integrated_rya_case(
    x=np.array(
        [40, 52, 7, 14]
    ),
    y=np.array(
        [9, 17, 54, 12]
    ),
    question="ry9",
    P_matrix=np.asarray(
        P_matrices["ry9"],
        dtype=float
    )
)

integrated_result_ry9

{'question': 'ry9',
 'n_X': 113,
 'n_Y': 92,
 'route': 'use_gamma_nn',
 'incompatibility_probability': 4.4769496425703704e-05,
 'threshold': 0.1,
 'validation_warnings': ['P_matrix row sums contain small rounding deviations (maximum error = 0.010000). Original matrix retained without renormalisation.'],
 'gamma_shape': 9.970475608021188,
 'gamma_rate': 20.926446972942113}

### 10.2 Successful end-to-end integration test

The integrated O1 screening and RYA neural-network pipeline was successfully
tested using the existing `ry9` example.

The compatibility detector estimated an incompatibility probability of
approximately 0.000045, which was well below the provisional routing threshold
of 0.10. The case was therefore routed to the existing Gamma neural network.

The integrated pipeline returned:

- route: `use_gamma_nn`;
- Gamma shape: 9.9705;
- Gamma rate: 20.9264.

These Gamma parameters are identical to the result obtained when the RYA neural
network was executed directly before integration.

The supplied `ry9` P matrix contains a small row-sum deviation of 0.01 due to
rounded probabilities. The integration retained the original matrix without
renormalisation and recorded this as a validation warning.

This confirms that the O1 compatibility layer can be inserted before the
existing RYA neural network without changing the original NN output for a case
that is routed as compatible.

In [108]:
incompatible_cases = (
    detector_expanded[
        detector_expanded[
            "compatibility_zone_v1"
        ] == "incompatible"
    ]
    .copy()
)

incompatible_X = incompatible_cases[
    feature_cols
]

incompatible_cases[
    "final_detector_probability"
] = loaded_detector.predict_proba(
    incompatible_X
)[:, 1]

test_incompatible = (
    incompatible_cases
    .sort_values(
        "final_detector_probability",
        ascending=False
    )
    .iloc[0]
)

print(
    "exp_code:",
    test_incompatible["exp_code"]
)

print(
    "question:",
    test_incompatible["P_matrix_question"]
)

print(
    "detector probability:",
    test_incompatible[
        "final_detector_probability"
    ]
)

print(
    "n_X:",
    test_incompatible["n_X"]
)

print(
    "n_Y:",
    test_incompatible["n_Y"]
)

print("\nX:")
print(
    [
        test_incompatible[f"X_{i}"]
        for i in range(1, 5)
    ]
)

print("\nY:")
print(
    [
        test_incompatible[f"Y_{i}"]
        for i in range(1, 5)
    ]
)

exp_code: orthog_B_3_large_classes_2166
question: ry24
detector probability: 0.9999998970793879
n_X: 6
n_Y: 5215

X:
[0.5, 0.166666666666667, 0.166666666666667, 0.166666666666667]

Y:
[0.187727708533078, 0.281495685522531, 0.172579098753595, 0.358197507190796]


### 10.3 Fallback-route integration test

A known high-confidence incompatible case from the Oct 9 validation dataset is
used to test the fallback branch of the integrated pipeline.

The selected case has a very large group-size imbalance:

- n_X = 6;
- n_Y = 5215;
- group-size ratio ≈ 869.

The final compatibility detector assigns this historical case an
incompatibility probability close to 1.0.

The exact instance-specific P matrix used during detector development is
retained for this integration test.

In [111]:
# Reconstruct raw counts from the stored proportions
x_incompatible = np.array(
    [3, 1, 1, 1],
    dtype=float
)

y_incompatible = np.array(
    [979, 1468, 900, 1868],
    dtype=float
)

print("x total:", x_incompatible.sum())
print("y total:", y_incompatible.sum())

print("\nReconstructed X proportions:")
print(
    x_incompatible
    / x_incompatible.sum()
)

print("\nStored X proportions:")
print(
    [
        test_incompatible[f"X_{i}"]
        for i in range(1, 5)
    ]
)

print("\nReconstructed Y proportions:")
print(
    y_incompatible
    / y_incompatible.sum()
)

print("\nStored Y proportions:")
print(
    [
        test_incompatible[f"Y_{i}"]
        for i in range(1, 5)
    ]
)

x total: 6.0
y total: 5215.0

Reconstructed X proportions:
[0.5        0.16666667 0.16666667 0.16666667]

Stored X proportions:
[0.5, 0.166666666666667, 0.166666666666667, 0.166666666666667]

Reconstructed Y proportions:
[0.18772771 0.28149569 0.1725791  0.35819751]

Stored Y proportions:
[0.187727708533078, 0.281495685522531, 0.172579098753595, 0.358197507190796]


In [113]:
P_incompatible = np.array([
    [
        test_incompatible["P_11"],
        test_incompatible["P_12"],
        test_incompatible["P_13"],
        test_incompatible["P_14"]
    ],
    [
        test_incompatible["P_21"],
        test_incompatible["P_22"],
        test_incompatible["P_23"],
        test_incompatible["P_24"]
    ],
    [
        test_incompatible["P_31"],
        test_incompatible["P_32"],
        test_incompatible["P_33"],
        test_incompatible["P_34"]
    ],
    [
        test_incompatible["P_41"],
        test_incompatible["P_42"],
        test_incompatible["P_43"],
        test_incompatible["P_44"]
    ]
], dtype=float)

print("Stored instance-specific P matrix:")
print(P_incompatible)

print("\nRow sums:")
print(
    P_incompatible.sum(axis=1)
)

Stored instance-specific P matrix:
[[0.463 0.262 0.166 0.109]
 [0.063 0.495 0.293 0.149]
 [0.009 0.131 0.563 0.298]
 [0.017 0.007 0.141 0.835]]

Row sums:
[1.    1.    1.001 1.   ]


In [115]:
integrated_result_incompatible = (
    run_integrated_rya_case(
        x=x_incompatible,
        y=y_incompatible,
        question="ry24",
        P_matrix=P_incompatible
    )
)

integrated_result_incompatible

{'question': 'ry24',
 'n_X': 6,
 'n_Y': 5215,
 'route': 'fallback_required',
 'incompatibility_probability': 0.9999998970793879,
 'threshold': 0.1,
 'validation_warnings': ['P_matrix row sums contain small rounding deviations (maximum error = 0.001000). Original matrix retained without renormalisation.'],
 'gamma_shape': None,
 'gamma_rate': None}

### 10.4 Integration routing verification

Both routing branches of the O1 integration pipeline were successfully tested.

For the `ry9` example, the compatibility detector produced an incompatibility
probability of approximately 0.000045, below the provisional threshold of
0.10. The case was routed to `use_gamma_nn`, and the existing RYA neural
network returned the same Gamma shape and rate values as the standalone
neural-network test.

For the high-risk `ry24` case, with group sizes of 6 and 5215, the detector
produced an incompatibility probability of approximately 0.9999999. The case
was routed to `fallback_required`, and the Gamma neural network was not used.

Small P-matrix row-sum deviations caused by rounded probabilities were retained
without renormalisation and reported as validation warnings.

These tests demonstrate that the O1 compatibility layer can safely intercept
high-risk cases while preserving the existing RYA neural-network behaviour for
cases routed as compatible.

## 11. Integrated pipeline validation suite

The integrated routing function is tested against representative valid and
invalid inputs.

The validation suite checks:

- a compatible-style case routed to the existing Gamma NN;
- a high-risk incompatible case routed to fallback;
- negative category counts;
- an empty comparison group;
- an invalid P-matrix shape;
- a P matrix with materially invalid row sums.

This provides a simple regression test for the Oct 16 integration milestone.

In [119]:
integration_test_results = []

def record_test(
    test_name,
    expected_route,
    result
):
    actual_route = result.get(
        "route",
        None
    )

    integration_test_results.append({
        "test_name": test_name,
        "expected_route": expected_route,
        "actual_route": actual_route,
        "passed": actual_route == expected_route,
        "reason": result.get("reason", ""),
        "incompatibility_probability":
            result.get(
                "incompatibility_probability",
                np.nan
            ),
        "gamma_shape":
            result.get(
                "gamma_shape",
                np.nan
            ),
        "gamma_rate":
            result.get(
                "gamma_rate",
                np.nan
            )
    })


# -------------------------------------------------
# Test 1: Safe ry9 case
# -------------------------------------------------

result_safe = run_integrated_rya_case(
    x=np.array(
        [40, 52, 7, 14],
        dtype=float
    ),
    y=np.array(
        [9, 17, 54, 12],
        dtype=float
    ),
    question="ry9",
    P_matrix=np.asarray(
        P_matrices["ry9"],
        dtype=float
    )
)

record_test(
    "safe_ry9",
    "use_gamma_nn",
    result_safe
)


# -------------------------------------------------
# Test 2: Known incompatible ry24 case
# -------------------------------------------------

result_fallback = run_integrated_rya_case(
    x=x_incompatible,
    y=y_incompatible,
    question="ry24",
    P_matrix=P_incompatible
)

record_test(
    "incompatible_ry24",
    "fallback_required",
    result_fallback
)


# -------------------------------------------------
# Test 3: Negative count
# -------------------------------------------------

result_negative = run_integrated_rya_case(
    x=np.array(
        [10, -1, 5, 2],
        dtype=float
    ),
    y=np.array(
        [8, 7, 4, 3],
        dtype=float
    ),
    question="ry9",
    P_matrix=np.asarray(
        P_matrices["ry9"],
        dtype=float
    )
)

record_test(
    "negative_count",
    "invalid_input",
    result_negative
)


# -------------------------------------------------
# Test 4: Zero-total group
# -------------------------------------------------

result_zero_group = run_integrated_rya_case(
    x=np.array(
        [0, 0, 0, 0],
        dtype=float
    ),
    y=np.array(
        [8, 7, 4, 3],
        dtype=float
    ),
    question="ry9",
    P_matrix=np.asarray(
        P_matrices["ry9"],
        dtype=float
    )
)

record_test(
    "zero_total_group",
    "invalid_input",
    result_zero_group
)


# -------------------------------------------------
# Test 5: Invalid P shape
# -------------------------------------------------

result_bad_shape = run_integrated_rya_case(
    x=np.array(
        [10, 8, 5, 2],
        dtype=float
    ),
    y=np.array(
        [8, 7, 4, 3],
        dtype=float
    ),
    question="test",
    P_matrix=np.eye(3)
)

record_test(
    "invalid_P_shape",
    "invalid_input",
    result_bad_shape
)


# -------------------------------------------------
# Test 6: Materially invalid P row sums
# -------------------------------------------------

P_bad_rows = np.array([
    [0.80, 0.10, 0.05, 0.01],
    [0.05, 0.80, 0.10, 0.05],
    [0.05, 0.10, 0.75, 0.10],
    [0.05, 0.05, 0.10, 0.80]
], dtype=float)

result_bad_rows = run_integrated_rya_case(
    x=np.array(
        [10, 8, 5, 2],
        dtype=float
    ),
    y=np.array(
        [8, 7, 4, 3],
        dtype=float
    ),
    question="test",
    P_matrix=P_bad_rows
)

record_test(
    "invalid_P_row_sums",
    "invalid_input",
    result_bad_rows
)


integration_test_df = pd.DataFrame(
    integration_test_results
)

display(integration_test_df)

,test_name,expected_route,actual_route,passed,reason,incompatibility_probability,gamma_shape,gamma_rate
0,safe_ry9,use_gamma_nn,use_gamma_nn,True,,0.000045,9.970476,20.926447
1,incompatible_ry24,fallback_required,fallback_required,True,,1.000000,NaN,NaN
2,negative_count,invalid_input,invalid_input,True,Negative category counts are not allowed.,NaN,NaN,NaN
3,zero_total_group,invalid_input,invalid_input,True,Both groups must contain at least one observat...,NaN,NaN,NaN
4,invalid_P_shape,invalid_input,invalid_input,True,P_matrix must be 4x4.,NaN,NaN,NaN
5,invalid_P_row_sums,invalid_input,invalid_input,True,P_matrix row sums deviate too far from 1. Maxi...,NaN,NaN,NaN


In [121]:
print(
    "Tests passed:",
    integration_test_df[
        "passed"
    ].sum(),
    "/",
    len(integration_test_df)
)

print(
    "\nFailed tests:",
    (
        ~integration_test_df[
            "passed"
        ]
    ).sum()
)

Tests passed: 6 / 6

Failed tests: 0


In [139]:
INTEGRATION_TEST_FILE = (
    BASE_DIR
    / "O1_07_integration_validation_results.csv"
)

integration_test_df.to_csv(
    INTEGRATION_TEST_FILE,
    index=False
)

print(
    "Saved:",
    INTEGRATION_TEST_FILE
)

Saved: C:\AIML Y2 S2\Capstone 1\RYA\O1_07_integration_validation_results.csv


## 12. Oct 16 integration milestone summary

The O1 compatibility detector was successfully integrated in front of the
existing RYA Gamma neural network.

### Completed integration components

- Reproduced the existing RYA `EXP002` neural-network pathway using the
  supplied 40-feature input definition, min-max metadata and ONNX model.
- Verified that the standalone RYA NN produces:
  - Gamma shape = 9.9705
  - Gamma rate = 20.9264
  for the existing `ry9` example.
- Verified the O1 screening feature implementation against historical
  detector-training data.
- Confirmed exact reproduction of the 54 detector features across 100
  randomly sampled historical cases.
- Retrained the final Logistic Regression detector using all 477
  high-confidence labelled cases:
  - 333 compatible;
  - 144 incompatible.
- Preserved the provisional Oct 9 high-recall routing threshold of 0.10.
- Saved the fitted detector and feature-order metadata.
- Implemented a single integrated routing function that performs:
  - input validation;
  - 54-feature compatibility screening;
  - probability-based routing;
  - fallback routing for high-risk cases;
  - existing RYA Gamma NN inference for cases routed as compatible.
- Confirmed both routing branches:
  - `ry9` -> `use_gamma_nn`;
  - high-risk `ry24` case -> `fallback_required`.
- Completed an integration validation suite with 6/6 tests passing.

### Key integration behaviour

The O1 detector does not replace or retrain the existing RYA Gamma neural
network.

Instead, it operates as a pre-inference screening layer.

Cases with estimated incompatibility probability below 0.10 continue to the
existing RYA neural network.

Cases with estimated incompatibility probability greater than or equal to
0.10 are intercepted and returned as `fallback_required`.

The current fallback is intentionally a placeholder. Development of the
alternative inference method belongs to the fallback component of the broader
project.

### P-matrix handling

Small P-matrix row-sum deviations were observed in supplied RYA matrices due
to rounded probability values.

The integration accepts small deviations up to 0.02, records a validation
warning, and retains the original matrix without renormalisation.

This preserves the existing neural-network input rather than silently changing
RYA's supplied probabilities.

### Current assumptions and limitations

The 0.10 detector threshold remains provisional and was selected from the
Oct 9 repeated out-of-fold analysis to prioritise recall of incompatible
cases.

The detector was fitted on 477 high-confidence labelled cases. Ambiguous Gamma
fit cases were excluded from detector training.

Historical detector-development instances contain instance-specific P
matrices, while the current RYA portal workflow appears to obtain a P matrix
from the question identifier. The production interface for P-matrix selection
should therefore be confirmed with RYA before deployment.

The current integrated implementation is a Python prototype. The existing RYA
Gamma model remains ONNX-compatible, while deployment packaging of the new O1
detector can be addressed after the integration interface is confirmed.

In [124]:
oct16_artifacts = [
    BASE_DIR / "O1_07_compatibility_detector_v1.joblib",
    BASE_DIR / "O1_07_compatibility_detector_v1_metadata.json",
    BASE_DIR / "O1_07_integration_validation_results.csv"
]

print("Oct 16 milestone artifacts:\n")

for path in oct16_artifacts:
    print(
        path.name,
        "->",
        "FOUND" if path.exists() else "MISSING"
    )

Oct 16 milestone artifacts:

O1_07_compatibility_detector_v1.joblib -> FOUND
O1_07_compatibility_detector_v1_metadata.json -> FOUND
O1_07_integration_validation_results.csv -> FOUND
